In [1]:
!pip install --no-index -U --find-links=/kaggle/input/tensorflow-2-15/tensorflow tensorflow==2.15.0
!pip install --no-index -U --find-links=/kaggle/input/deeptables-v0-2-5/deeptables-0.2.5 deeptables==0.2.5
!pip install --no-index -U --find-links=/kaggle/input/fix-deeptables/deeptables-0.2.6 deeptables==0.2.6

Looking in links: /kaggle/input/tensorflow-2-15/tensorflow
Processing /kaggle/input/tensorflow-2-15/tensorflow/tensorflow-2.15.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/tensorflow-2-15/tensorflow/ml_dtypes-0.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from tensorflow==2.15.0)
Processing /kaggle/input/tensorflow-2-15/tensorflow/wrapt-1.14.1-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from tensorflow==2.15.0)
Processing /kaggle/input/tensorflow-2-15/tensorflow/tensorboard-2.15.1-py3-none-any.whl (from tensorflow==2.15.0)
Processing /kaggle/input/tensorflow-2-15/tensorflow/keras-2.15.0-py3-none-any.whl (from tensorflow==2.15.0)
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.16.0
    Uninstalling wrapt-1.16.0:
      Successfully uninstalled wrapt-1.16.0
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalli

In [2]:
import numpy as np
import polars as pl
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.pyplot as plt

import gc
import os
import sys
import io
import itertools
import joblib

from tqdm import tqdm
from IPython.display import clear_output

import warnings
warnings.filterwarnings('ignore')
import kaggle_evaluation.mcts_inference_server
import mcts_inference_server

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import *
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import VotingRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from deeptables.models import DeepTable

2024-11-18 00:03:43.986521: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-18 00:03:43.986618: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-18 00:03:43.989067: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
test = pl.read_csv('/kaggle/input/um-game-playing-strength-of-mcts-variants/test.csv')
train = pl.read_csv('/kaggle/input/um-game-playing-strength-of-mcts-variants/train.csv')
concept = pl.read_csv('/kaggle/input/um-game-playing-strength-of-mcts-variants/concepts.csv')
submission = pl.read_csv('/kaggle/input/um-game-playing-strength-of-mcts-variants/sample_submission.csv')

In [4]:
# Englishruleから特徴量作成
# def extract_from_engrule(df,tfidf,svd):
#     tfidf_matrix = tfidf.transform(df['EnglishRules'])
#     svd_matrix = svd.transform(tfidf_matrix)
#     svd_df = pd.DataFrame(svd_matrix, columns=[f'svd_component_{i+1}' for i in range(10)])
#     train = df.to_pandas()
#     train = pd.concat([train, svd_df], axis=1)
#     train = pl.from_pandas(train)

#     return train

In [5]:
# GameRulesetNameからゲーム名で特徴量を作成する
def extract_game_name(df):
    # 文字列長
    df = df.with_columns(pl.col("GameRulesetName").str.len_chars().alias("GameRulesetName_len"))
    # _で区切られた単語数
    df = df.with_columns(pl.col("GameRulesetName").str.count_matches("_").alias("GameRulesetName_count"))
    #アルファベットと数字の割合
    # train = train.with_columns([
    #     pl.col("GameRulesetName").str.count_match(r"[A-Za-z]").alias("alpha_count"),
    #     pl.col("GameRulesetName").str.count_match(r"\d").alias("digit_count")
    # ])

    # 頻出単語があるかないかフラグ
    frequent_word = ["Suggested"]
    for keyword in frequent_word:
        df = df.with_columns([
            pl.col("GameRulesetName").str.contains(keyword).cast(pl.Int8).alias(f"contains_{keyword.lower()}")
        ])
    frequent_word_column = [f"contains_{keyword.lower()}" for keyword in frequent_word]

    return df

frequent_word = ["Suggested"]
frequent_word_column = [f"contains_{keyword.lower()}" for keyword in frequent_word]

In [6]:
nomean_cols = [] 
for col,dtype in train.schema.items():
    # 列ですべての値が等しいカラムについては取り除く
    if dtype != pl.Utf8:
        if train[col].n_unique() == 1:
            nomean_cols.append(col)
    # nullしかないカラムも取り除く
    elif train[col].is_null().sum() == len(train):
        nomean_cols.append(col)

print(len(nomean_cols))

216


In [7]:
# 語尾にfrequencyをつけると同じカラムが出現するやつ
freq_df = pd.read_csv(io.StringIO('''
VoteDecision
SwapPlayersDecision
PassDecision
ProposeDecision
AddDecision
PromotionDecision
RemoveDecision
RotationDecision
StepDecision
StepDecisionToEmpty
StepDecisionToFriend
StepDecisionToEnemy
SlideDecision
SlideDecisionToEmpty
SlideDecisionToEnemy
SlideDecisionToFriend
LeapDecision
LeapDecisionToEmpty
LeapDecisionToEnemy
HopDecision
HopDecisionMoreThanOne
HopDecisionEnemyToEmpty
HopDecisionFriendToEmpty
HopDecisionEnemyToEnemy
HopDecisionFriendToEnemy
FromToDecision
FromToDecisionEmpty
FromToDecisionEnemy
FromToDecisionFriend
SwapPiecesDecision
ShootDecision
VoteEffect
SwapPlayersEffect
PassEffect
Roll
ProposeEffect
AddEffect
Sow
SowCapture
SowRemove
SowBacktracking
PromotionEffect
RemoveEffect
PushEffect
Flip
SetNextPlayer
MoveAgain
SetValue
SetCount
SetRotation
StepEffect
SlideEffect
LeapEffect
HopEffect
FromToEffect
ReplacementCapture
HopCapture
HopCaptureMoreThanOne
DirectionCapture
EncloseCapture
CustodialCapture
InterveneCapture
SurroundCapture
CaptureSequence
LineEnd
LineWin
LineLoss
LineDraw
ConnectionEnd
ConnectionWin
ConnectionLoss
GroupEnd
GroupWin
GroupLoss
GroupDraw
LoopEnd
LoopWin
LoopLoss
PatternEnd
PatternWin
PathExtentEnd
PathExtentWin
PathExtentLoss
TerritoryEnd
TerritoryWin
Checkmate
CheckmateWin
NoTargetPieceEnd
NoTargetPieceWin
EliminatePiecesEnd
EliminatePiecesWin
EliminatePiecesLoss
EliminatePiecesDraw
NoOwnPiecesEnd
NoOwnPiecesWin
NoOwnPiecesLoss
FillEnd
FillWin
ReachEnd
ReachWin
ReachLoss
ReachDraw
ScoringEnd
ScoringWin
ScoringLoss
ScoringDraw
NoMovesEnd
NoMovesWin
NoMovesLoss
NoMovesDraw
NoProgressEnd
NoProgressDraw
Draw
'''), header=None, )
freq_cols = list(freq_df.iloc[:,0])
print(len(freq_cols))

113


In [8]:
duplicate_cols =['AsymmetricForces','AsymmetricPiecesType','PieceDirection','Team','SpiralTiling','CircleTiling','ShibumiStyle','MancalaStyle','NumPerimeterSites', 'SwapOption',
                 'SetRotationFrequency','PathExtent','SowOriginFirst','SetRotationFrequency','LeftwardDirection','LeftwardsDirection','ForwardLeftDirection','BackwardLeftDirection','LoopEndFrequency',
                 'PatternEndFrequency','TerritoryEndFrequency','NoProgressEndFrequency''StateType']

In [9]:
%%time

def remove_useless(df: pl.DataFrame, nomean_cols : list, freq_cols : list, duplicate_cols)  -> pl.DataFrame:

    #追加でいらないやつを取り除く
    drop_cols = nomean_cols + freq_cols + ["GameRulesetName","Id",]+duplicate_cols

    Dropped = [col for col in drop_cols if col in df.columns]

    df = df.drop(Dropped)

    return df

CPU times: user 9 µs, sys: 0 ns, total: 9 µs
Wall time: 12.6 µs


In [10]:
# 一塊にしてワンホットを一つの列にしたほうがよさそうなやつ
shape_list = [ 'Shape',
 'SquareShape',
 'HexShape',
 'TriangleShape',
 'DiamondShape',
 'RectangleShape',
 'SpiralShape',
 'CircleShape',
 'StarShape',
 'SquarePyramidalShape',
#  'RegularShape',
#  'PolygonShape',
]

tiling_list = ["Tiling",
 'SquareTiling',
 'HexTiling',
 'TriangleTiling',
 'SemiRegularTiling',
 'MorrisTiling',
#  'CircleTiling',
#  'ConcentricTiling',
#  'SpiralTiling',
 'AlquerqueTiling',]

mancala_list = [ 'MancalaBoard',
#  'MancalaStores',
 'MancalaTwoRows',
 'MancalaThreeRows',
 'MancalaFourRows',
 'MancalaSixRows',
 'MancalaCircular']

alquerque_list = [ 'AlquerqueBoard',
 'AlquerqueBoardWithOneTriangle',
 'AlquerqueBoardWithTwoTriangles',
 'AlquerqueBoardWithFourTriangles',
 'AlquerqueBoardWithEightTriangles',] 

board_list = [ 'ThreeMensMorrisBoard',
 'ThreeMensMorrisBoardWithTwoTriangles',
 'NineMensMorrisBoard',
 'StarBoard',
 'CrossBoard',
 'KintsBoard',
 'PachisiBoard',
 'FortyStonesWithFourGapsBoard']

dice_list = [  'Dice',
 'DiceD2',
 'DiceD4',
 'DiceD6',]

category_list = [shape_list , tiling_list ,mancala_list , alquerque_list ,board_list ,dice_list]
addcat_list = ["Shape_category","Tiling_category","Mancala_category","Alquerque_category","Board_category","Dice_category"]
def create_category(df,category_list, addcat_list):
    for i,col_list in enumerate(category_list):
        addcat_name = addcat_list[i]
        df = df.with_columns(pl.lit("none").alias(addcat_name))
        for col in col_list:
            df = df.with_columns(pl.col(col).cast(pl.Utf8))
            df = df.with_columns(pl.col(col).replace(1, col).alias(col))
            df = df.with_columns(
                      pl.when(pl.col(col) == col).then(pl.col(col)).otherwise(pl.col(addcat_name)).alias(addcat_name)
                  )
        df = df.drop(col_list)
    return df

train = create_category(train,category_list, addcat_list)

In [11]:
# ゲームの複雑さを表現したい
def create_category_sum(df):
    directions_col = ['Directions','AbsoluteDirections','AllDirections','AdjacentDirection','OrthogonalDirection','DiagonalDirection','RotationalDirection','SameLayerDirection','RelativeDirections',
                      'ForwardDirection','BackwardDirection','ForwardsDirection','BackwardsDirection','ForwardLeftDirection','BackwardLeftDirection','SameDirection','OppositeDirection',]

    style_cols = [ 'Style','BoardStyle','GraphStyle','ChessStyle','GoStyle','PenAndPaperStyle',
                   'BackgammonStyle','JanggiStyle','XiangqiStyle','ShogiStyle','TableStyle','SurakartaStyle','TaflStyle',]

    # component_cols = ['ComponentStyle','AnimalComponent','ChessComponent','FairyChessComponent','PloyComponent','ShogiComponent','XiangqiComponent',
    #                   'StrategoComponent','JanggiComponent','CheckersComponent','BallComponent','TaflComponent','DiscComponent','MarkerComponent']

    spacecondition_cols = ['Line','Connection','Group','Contains','Loop','Pattern','Territory','Fill','Distance',]

    checc_cols = ["KingComponent","QueenComponent","KnightComponent","RookComponent","BishopComponent","PawnComponent"]

    logic_cols = ["Logic", "Conjunction", "Disjunction",  "Negation"]


    df = df.with_columns(sum([pl.col(c) for c in directions_col]).alias("DirectionSum"))
    df = df.with_columns(sum([pl.col(c) for c in style_cols]).alias("StyleSum"))
    # df = df.with_columns(sum([pl.col(c) for c in component_cols]).alias("ComponentSum"))
    df = df.with_columns(sum([pl.col(c) for c in spacecondition_cols]).alias("SpaceConditionSum"))
    df = df.with_columns(sum([pl.col(c) for c in checc_cols]).alias("CheccSum"))
    df = df.with_columns(sum([pl.col(c) for c in logic_cols]).alias("LogicSum"))


    # df = df.drop(directions_col)
    # # df = df.drop(style_cols)
    # # df = df.drop(component_cols)
    # df = df.drop(spacecondition_cols)
    # df = df.drop(checc_cols)
    # df = df.drop(logic_cols)



    return df


In [12]:
def drop_col_almostnomean(train):
    check = ['Cooperation',
 'SpiralShape',
 'StarShape',
 'MancalaSixRows',
 'MancalaCircular',
 'AlquerqueBoardWithEightTriangles',
 'StarBoard',
 'KintsBoard',
 'PachisiBoard',
 'FortyStonesWithFourGapsBoard',
 'NumLayers',
 'PieceRotation',
 'TurnKo',
 'AutoMove',
 'InitialRandomPlacement',
 'Moves',
 'SwapPlayersDecisionFrequency',
 'ProposeDecisionFrequency',
 'RotationDecisionFrequency',
 'SlideDecisionToFriendFrequency',
 'HopDecisionFriendToFriendFrequency',
 'HopDecisionEnemyToEnemyFrequency',
 'ShootDecisionFrequency',
 'ProposeEffectFrequency',
 'PushEffectFrequency',
 'FlipFrequency',
 'SetCountFrequency',
 'DirectionCaptureFrequency',
 'InterveneCaptureFrequency',
 'SurroundCaptureFrequency',
 'SameLayerDirection',
 'RightwardsDirection',
 'ConnectionLossFrequency',
 'GroupEndFrequency',
 'GroupWinFrequency',
 'LoopWinFrequency',
 'PatternWinFrequency',
 'EliminatePiecesLossFrequency',
 'NoOwnPiecesLossFrequency',
 'ReachLossFrequency',
 'ReachDrawFrequency',
 'ScoringLossFrequency',
 'NoMovesLossFrequency',
 'Absolute',
 'Exponentiation',
 'JanggiStyle',
 'SurakartaStyle',
 'PloyComponent']
    train = train.drop(check,axis=1)
    return train

In [13]:
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

def symbol_match(rule):
    stack = []
    data = []
    for i in range(len(rule)):
        if rule[i] in ['(', '{']:
            stack.append(rule[i])
        elif rule[i] in [')', '}']:
            stack = stack[:-1]
        elif (rule[i] == '"') and (len(stack) > 0) and (stack[-1] == '"'):
            stack = stack[:-1]
        elif rule[i] == '"':
            stack.append('"')
        data.append(rule[i])
        if len(stack) == 0:
            return ''.join(data).strip(), rule[i + 1:].strip()
    return '', ''

def get_ruledata(rule):
    rule = rule[len('(game '):-1].strip()
    datas = []
    while len(rule):
        data, rule = symbol_match(rule)
        datas.append(data)
    return datas

def process_rules(rules, chunk_size=100):
    with Pool(cpu_count()) as pool:
        # tqdm for progress bar, use imap_unordered for faster return of results
        result = list(pool.imap(get_ruledata, rules, chunksize=chunk_size))
    return result

def split_rule(df):
    df = df.to_pandas()

    rules = df['LudRules'].values
    rule_datas = process_rules(rules)
    rule_datas = np.array(rule_datas, dtype=object)
    rule_datas = pd.DataFrame(rule_datas,columns =['game', 'players', 'equipment', 'rules'] )
    rule_datas = rule_datas.drop(columns=["game","equipment","rules"])
    df = pd.concat([df, rule_datas], axis=1)
    df = pl.from_pandas(df)

    return df

In [14]:
def parse_prefix_tree(s):
    stack = []
    indent = 0
    current_line = ''
    indent_box_equipment = []
    indent_box_phase = []
    indent_box_rule_start = []
    indent_box_rule_play = []
    indent_box_rule_end = []
    flag = ""

    def flush_line():
        nonlocal current_line,indent_box_equipment,indent_box_phase,indent_box_rule_start,indent_box_rule_play,indent_box_rule_end,flag
        if current_line.strip():
            # print(f'Level {indent:02} - ' + '. ' * indent, current_line.strip())
            if indent == 2 and current_line.strip() == "equipment":
                  flag = "equipment"
            if indent == 2 and current_line.strip() == "phases:":
                  flag = "phases"
            if indent == 3 and current_line.strip() == "start":
                  flag = "start"
            if indent == 3 and current_line.strip() == "play":
                  flag = "play"
            if indent == 3 and current_line.strip() == "end":
                  flag  = "end"

            if flag == "equipment":
                  indent_box_equipment.append(indent)
            elif flag == "phases":
                  indent_box_phase.append(indent)
            elif flag == "start":
                  indent_box_rule_start.append(indent)
            elif flag == "play":
                  indent_box_rule_play.append(indent)
            elif flag == "end":
                  indent_box_rule_end.append(indent)
        current_line = ''
    i = 0
    while i < len(s):
        char = s[i]
        if char == '(' or char == '{':
            flush_line()
            stack.append(char)
            indent += 1
        elif char == ')' or char == '}':
            flush_line()
            indent -= 1
        elif char not in '(){}':
            j = i
            while j < len(s) and s[j] not in '(){}':
                j += 1
            current_line += s[i:j].strip()
            i = j - 1
        if current_line and (char == ')' or char == '}'):
            flush_line()
        i += 1
    flush_line()

    if len(indent_box_phase) == 0:
        indent_box_phase.append(0)
    if len(indent_box_rule_start) == 0:
        indent_box_rule_start.append(0)
    if len(indent_box_rule_play) == 0:
        indent_box_rule_play.append(0)
    if len(indent_box_rule_end) == 0:
        indent_box_rule_end.append(0)

    return [max(indent_box_equipment),np.mean(indent_box_equipment),len(indent_box_equipment) ,max(indent_box_phase),np.mean(indent_box_phase),len(indent_box_phase)
    ,max(indent_box_rule_start),np.mean(indent_box_rule_start),len(indent_box_rule_start),max(indent_box_rule_play),np.mean(indent_box_rule_play),len(indent_box_rule_play),max(indent_box_rule_end),np.mean(indent_box_rule_end),len(indent_box_rule_end)]

In [15]:
# ルールから特徴量を作成
def make_rule_features(df):
    row_list = []
    for i in df["LudRules"]:
        row_list.append(parse_prefix_tree(i))

    LudRules_df = pd.DataFrame(row_list,columns=["LudRules_equipment_max","LudRules_equipment_mean","LudRules_equipment_num","LudRules_phase_max","LudRules_phase_mean","LudRules_phase_num","LudRules_rule_start_max","LudRules_rule_start_mean","LudRules_rule_start_num",
                                                 "LudRules_rule_play_max","LudRules_rule_play_mean","LudRules_rule_play_num","LudRules_rule_end_max","LudRules_rule_end_mean","LudRules_rule_end_num"])

    df = df.to_pandas()
    df = pd.concat([df,LudRules_df],axis=1)
    df = df.drop(["EnglishRules","LudRules"],axis = 1)
    df = pl.from_pandas(df)

    return df

In [16]:
%%time

do_cat_col = [
    'Stochastic','Asymmetric','AsymmetricForces','OpeningContract','Repetition','Phase','Draw',
]
def process_agent_cols(df):

    df = df.with_columns(
        pl.col('agent1').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 1).alias('p1_selection'),
        pl.col('agent1').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 2).alias('p1_exploration').cast(pl.Float32),
        pl.col('agent1').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 3).alias('p1_playout'),
        pl.col('agent1').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 4).alias('p1_bounds'),
        pl.col('agent2').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 1).alias('p2_selection'),
        pl.col('agent2').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 2).alias('p2_exploration').cast(pl.Float32),
        pl.col('agent2').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 3).alias('p2_playout'),
        pl.col('agent2').str.extract(r'MCTS-(.*)-(.*)-(.*)-(.*)', 4).alias('p2_bounds')
    ).with_columns(
        pl.concat_str('p1_selection', 'p2_selection', separator=' - ').alias('p_selection'),
        pl.concat_str('p1_playout', 'p2_playout', separator=' - ').alias('p_playout'),
        pl.concat_str('p1_exploration', 'p2_exploration', separator=' - ').alias('p_exploration'),
        pl.concat_str('p1_bounds', 'p2_bounds', separator=' - ').alias('p_bounds'),
    )

    df = df.drop(["agent1","agent2"])

    df = df.with_columns(
        [pl.col(col).cast(pl.Utf8).cast(pl.Categorical) for col in df.columns if col in ['p1_selection','p1_playout','p1_bounds','p2_selection','p2_playout','p2_bounds','p_selection', 'p_exploration', "p_bounds",'p_playout',"players"]+addcat_list+frequent_word_column]
    )

    df = df.with_columns(
        [pl.col(col).cast(pl.Float32) for col in df.columns if col not in ['p1_selection','p1_playout','p1_bounds','p2_selection','p2_playout','p2_bounds','p_selection', 'p_exploration', "p_bounds", 'p_playout',"players"]+addcat_list+frequent_word_column]
    )


    return df.to_pandas()

CPU times: user 4 µs, sys: 2 µs, total: 6 µs
Wall time: 10.5 µs


In [17]:
def add_agentcol(df):
    df["same_selection"] = (df["p1_selection"].astype("str") == df["p2_selection"].astype("str")).astype("category")
    df["diff_exploration"] = (df["p1_exploration"] - df["p2_exploration"]).astype("float32")
    df["same_playout"] = (df["p1_playout"].astype("str") == df["p2_playout"].astype("str")).astype("category")
    df["same_bounds"] = (df["p1_bounds"].astype("str") == df["p2_bounds"].astype("str")).astype("category")
    df["same_exploration"] = (df["p1_exploration"].astype("str") == df["p2_exploration"].astype("str")).astype("category")
    df["agent_pair_fit"] = df["same_selection"].astype("int") + df["same_playout"].astype("int") + df["same_exploration"].astype("int") + df["same_bounds"].astype("int")
    df = df.drop(["same_selection","same_playout","same_bounds","same_exploration"],axis=1)
    df["p1_selection_playout"] = (df["p1_selection"].astype("str") + " -" +df["p1_playout"].astype("str")).astype("category")
    df["p2_selection_playout"] = (df["p2_selection"].astype("str") + " -" +df["p2_playout"].astype("str")).astype("category")
    df["p1_selection_exploration"] = (df["p1_selection"].astype("str") + " -" +df["p1_exploration"].astype("str")).astype("category")
    df["p2_selection_exploration"] = (df["p2_selection"].astype("str") + " -" +df["p2_exploration"].astype("str")).astype("category")
    # df["p1_selection_bounds"] = (df["p1_selection"].astype("str") + " -" +df["p1_bounds"].astype("str")).astype("category")
    # df["p2_selection_bounds"] = (df["p2_selection"].astype("str") + " -" +df["p2_bounds"].astype("str")).astype("category")
    # df["p1_playout_exploration"] = (df["p1_playout"].astype("str") + " -" +df["p1_exploration"].astype("str")).astype("category")
    # df["p2_playout_exploration"] = (df["p2_playout"].astype("str") + " -" +df["p2_exploration"].astype("str")).astype("category")
    # df["p1_exploration_bounds"] = (df["p1_exploration"].astype("str") + " -" +df["p1_bounds"].astype("str")).astype("category")
    # df["p2_exploration_bounds"] = (df["p2_exploration"].astype("str") + " -" +df["p2_bounds"].astype("str")).astype("category")
    # df["p1_playout_bounds"] = (df["p1_playout"].astype("str") + " -" +df["p1_bounds"].astype("str")).astype("category")
    # df["p2_playout_bounds"] = (df["p2_playout"].astype("str") + " -" +df["p2_bounds"].astype("str")).astype("category")
    # df["p_selection_exploration"] = (df["p1_selection_exploration"].astype("str") + " -" +df["p2_selection_exploration"].astype("str")).astype("category")
    # df["p_selection_playout"] = (df["p1_selection_playout"].astype("str") + " -" +df["p2_selection_playout"].astype("str")).astype("category")
    # #df["p_selection_bounds"] = (df["p_selection"].astype("str") + " -" +df["p_bounds"].astype("str")).astype("category")
    # df["p_playout_exploration"] = (df["p1_playout_exploration"].astype("str") + " -" +df["p2_playout_exploration"].astype("str")).astype("category")
    # #df["p_playout_bounds"] = (df["p_playout"].astype("str") + " -" +df["p_bounds"].astype("str")).astype("category")
    # df["same_selection_playout"] = (df["p1_selection_playout"].astype("str") == df["p2_selection_playout"].astype("str")).astype("category")
    df["same_selection_exploration"] = (df["p1_selection_exploration"].astype("str") == df["p2_selection_exploration"].astype("str")).astype("category")
    # df["same_selection_bounds"] = (df["p1_selection_bounds"].astype("str") == df["p2_selection_bounds"].astype("str")).astype("category")
    # df["same_playout_exploration"] = (df["p1_playout_exploration"].astype("str") == df["p2_playout_exploration"].astype("str")).astype("category")
    # df["same_playout_bounds"] = (df["p1_playout_bounds"].astype("str") == df["p2_playout_bounds"].astype("str")).astype("category")
    # df = df.drop(["p1_selection","p1_playout","p2_selection","p2_playout"],axis = 1)
    return df


In [18]:
def drop_cols_conceptwise(df, concepts):

    concepts = concepts.drop(['Id', 'Description', 'TypeId', 'DataTypeId', 'ComputationTypeId', 'LeafNode', 'ShowOnWebsite'])
    concepts = concepts.with_columns([pl.col(col).cast(pl.String) for col in concepts.columns])

    concepts = concepts.filter(pl.col('TaxonomyString').str.starts_with('2.2.5.1') |
                                pl.col('TaxonomyString').str.starts_with('2.2.5.2') |
                                pl.col('TaxonomyString').str.starts_with('2.2.5.3') |
                                pl.col('TaxonomyString').str.starts_with('2.2.5.4') |
                                pl.col('TaxonomyString').str.starts_with('2.2.5.5') |
                                pl.col('TaxonomyString').str.starts_with('5.1') 
#                                 pl.col('TaxonomyString').str.starts_with('3.3.1') |
#                                 pl.col('TaxonomyString').str.starts_with('3.3.2') |
#                                 pl.col('TaxonomyString').str.starts_with('3.4.1') | 
#                                 pl.col('TaxonomyString').str.starts_with('3.4.2')
                                )

    concept_cols = concepts['Name'].to_list()

    return df.drop([col for col in concept_cols if col in df.columns])

In [19]:
def feature_engineering(df):
    col42 = ["DurationActions","DurationMoves","DurationTurns","DurationTurnsStdDev","DurationTurnsNotTimeouts"]
    col43 = ["DecisionMoves","GameTreeComplexity","StateTreeComplexity"]
    col45 = ["AdvantageP1","Balance","Completion","Drawishness","Timeouts","OutcomeUniformity"]
    col72 = ["PlayoutsPerSecond","MovesPerSecond"]
    cols = [col42,col43,col45,col72]
    for col in cols:
      for pair in itertools.combinations(col, 2):
          col_name = f"{pair[0]}_{pair[1]}_ratio"
          df[col_name] = df[pair[0]]/(df[pair[1]]+1e-5)
          df[col_name] = df[col_name].astype("float32")
          col_name = f"{pair[0]}_{pair[1]}_diff"
          df[col_name] = df[pair[0]]-df[pair[1]]
          df[col_name] = df[col_name].astype("float32")


    for cols in [col42,col43]:
      for col in cols:
        for b_col in col72:
          col_name = f"{col}_{b_col}_ratio"
          df[col_name] = df[col]/(df[b_col]+1e-5)
          df[col_name] = df[col_name].astype("float32")
    return df

In [20]:
# 正規化
def normalize_columns(df):
    # df = df.with_columns([
    #     (pl.col("AdvantageP1") * 2 - 1).alias("AdvantageP1")
    # ])
    df["AdvantageP1"]  = df["AdvantageP1"]*2-1
    return df

In [21]:
def get_row_feature(df):
    df = df.to_pandas()
    df["row_count_0"] = (df.select_dtypes(include=["number"]) == 0).sum(axis=1)
    df["row_count_1"] = (df.select_dtypes(include=["number"]) == 1).sum(axis=1)
    df["row_count_under0"] = (df.select_dtypes(include=["number"]) < 0).sum(axis=1)
    df = pl.DataFrame(df)
    return df

In [22]:
def feature_domain(train):
    train["fit_OutcomeUniformity_Balance_diff"] = (train["OutcomeUniformity"]*train["OutcomeUniformity"]*0.48-1.3*train["OutcomeUniformity"]+0.85-train["Balance"])
    # train["fit_OutcomeUniformity_Balance_diff"] = train["fit_OutcomeUniformity_Balance_diff"].apply(lambda x : 0 if x == 0 else 1)
    train["fit_AdvantageP1_OutcomeUniformity_diff"] = ((train["OutcomeUniformity"] - (train["AdvantageP1"]*train["AdvantageP1"]*4 - 4*train["AdvantageP1"] + 1))*10)
    # train["fit_AdvantageP1_OutcomeUniformity_diff"] = train["fit_AdvantageP1_OutcomeUniformity_diff"].apply(lambda x : 0 if x == 0 else 1)
    train["fit_AdvantageP1_Balance_diff"] = (train["Balance"] - (-np.abs(train["AdvantageP1"]-0.5)*2+1))
    # train["fit_AdvantageP1_Balance_diff"] = train["fit_AdvantageP1_Balance_diff"].apply(lambda x : 0 if x == 0 else 1)
    return train

In [23]:
# ある特定のゲームにしか登場しない特徴量(最大要素を除いた要素の数の合計が13個以下)
def drop_col_almostnomean(train):
    check = ['Cooperation',
 'TriangleShape',
 'DiamondShape',
 'SpiralShape',
 'StarShape',
 'SquarePyramidalShape',
 'SemiRegularTiling',
 'MancalaThreeRows',
 'MancalaSixRows',
 'MancalaCircular',
 'AlquerqueBoardWithOneTriangle',
 'AlquerqueBoardWithFourTriangles',
 'AlquerqueBoardWithEightTriangles',
 'ThreeMensMorrisBoard',
 'ThreeMensMorrisBoardWithTwoTriangles',
 'NineMensMorrisBoard',
 'StarBoard',
 'KintsBoard',
 'PachisiBoard',
 'FortyStonesWithFourGapsBoard',
 'Boardless',
 'NumOffDiagonalDirections',
 'NumLayers',
 'Piece',
 'PieceRotation',
 'LargePiece',
 'TurnKo',
 'AutoMove',
 'InitialRandomPlacement',
 'InitialCost',
 'Moves',
 'SwapPlayersDecisionFrequency',
 'ProposeDecisionFrequency',
 'RotationDecisionFrequency',
 'StepDecisionToFriendFrequency',
 'SlideDecisionToFriendFrequency',
 'HopDecisionFriendToFriendFrequency',
 'HopDecisionEnemyToEnemyFrequency',
 'HopDecisionFriendToEnemyFrequency',
 'FromToDecisionFrequency',
 'ShootDecisionFrequency',
 'ProposeEffectFrequency',
 'PushEffectFrequency',
 'FlipFrequency',
 'SetCountFrequency',
 'MaxDistance',
 'DirectionCaptureFrequency',
 'EncloseCaptureFrequency',
 'InterveneCaptureFrequency',
 'SurroundCaptureFrequency',
 'Loop',
 'Pattern',
 'Territory',
 'RotationalDirection',
 'SameLayerDirection',
 'BackwardsDirection',
 'RightwardsDirection',
 'SameDirection',
 'OppositeDirection',
 'LineLossFrequency',
 'ConnectionLossFrequency',
 'GroupEndFrequency',
 'GroupWinFrequency',
 'LoopWinFrequency',
 'PatternWinFrequency',
 'TerritoryWinFrequency',
 'NoTargetPieceWinFrequency',
 'EliminatePiecesLossFrequency',
 'EliminatePiecesDrawFrequency',
 'NoOwnPiecesLossFrequency',
 'FillEndFrequency',
 'FillWinFrequency',
 'ReachLossFrequency',
 'ReachDrawFrequency',
 'ScoringLossFrequency',
 'NoMovesLossFrequency',
 'NoMovesDrawFrequency',
 'BoardSitesOccupiedChangeNumTimes',
 'BranchingFactorChangeNumTimesn',
 'PieceNumberChangeNumTimes',
 'Absolute',
 'Exponentiation',
 'JanggiStyle',
 'XiangqiStyle',
 'TableStyle',
 'SurakartaStyle',
 'TaflStyle',
 'NoBoard',
 'PloyComponent',
 'ShogiComponent',
 'XiangqiComponent',
 'StrategoComponent',
 'JanggiComponent',
 'TaflComponent',
 'ShowPieceValue',
 'ShowPieceState',
 'VisitedSites']
    train = train.drop(check ,axis=1)
    return train

In [24]:
def get_target_encoding_cluster_test(test,target_list):
    # target_cols = ['utility_agent1','num_losses_agent1','num_wins_agent1', 'num_draws_agent1']
    target_col = 'utility_agent1'
    test[f"cluster_{target_col}_encoded_std"] = test["cluster"].map(target_list[0])
    test[f"cluster_{target_col}_encoded_std"] = test[f"cluster_{target_col}_encoded_std"].astype("float32")
    test[f"cluster_{target_col}_encoded_iqr"] = test["cluster"].map(target_list[1])
    test[f"cluster_{target_col}_encoded_iqr"] = test[f"cluster_{target_col}_encoded_iqr"].astype("float32")
    test[f"cluster_{target_col}_encoded_median"] = test["cluster"].map(target_list[2])
    test[f"cluster_{target_col}_encoded_median"] = test[f"cluster_{target_col}_encoded_median"].astype("float32")
    test[f"cluster_{target_col}_encoded_kurtosis"] = test["cluster"].map(target_list[3])
    test[f"cluster_{target_col}_encoded_kurtosis"] = test[f"cluster_{target_col}_encoded_kurtosis"].astype("float32")
    test[f"cluster_{target_col}_encoded_skewness"] = test["cluster"].map(target_list[4])
    test[f"cluster_{target_col}_encoded_skewness"] = test[f"cluster_{target_col}_encoded_skewness"].astype("float32")
    # test[f"cluster_{target_col}_encoded_mode"] = test["cluster"].map(target_list[i][5])
    # test[f"cluster_{target_col}_encoded_mode"] = test[f"cluster_{target_col}_encoded_mode"].astype("float32")
    test[f"cluster_{target_col}_encoded_1ratio"] = test["cluster"].map(target_list[5])
    test[f"cluster_{target_col}_encoded_1ratio"] = test[f"cluster_{target_col}_encoded_1ratio"].astype("float32")
    test[f"cluster_{target_col}_encoded_0ratio"] = test["cluster"].map(target_list[6])
    test[f"cluster_{target_col}_encoded_0ratio"] = test[f"cluster_{target_col}_encoded_0ratio"].astype("float32")
    test[f"cluster_{target_col}_encoded_11ratio"] = test["cluster"].map(target_list[7])
    test[f"cluster_{target_col}_encoded_11ratio"] = test[f"cluster_{target_col}_encoded_11ratio"].astype("float32")
    return test

In [25]:
cluster_column = [
 'Stochastic',
 'Asymmetric',
 'PlayersWithDirections',
 'Cooperation',
 'Shape',
 'SquareShape',
 'HexShape',
 'TriangleShape',
 'DiamondShape',
 'RectangleShape',
 'SpiralShape',
 'CircleShape',
 'StarShape',
 'SquarePyramidalShape',
 'RegularShape',
 'PolygonShape',
 'Tiling',
 'SquareTiling',
 'HexTiling',
 'TriangleTiling',
 'SemiRegularTiling',
 'MorrisTiling',
 'ConcentricTiling',
 'AlquerqueTiling',
 'MancalaBoard',
 'MancalaStores',
 'MancalaTwoRows',
 'MancalaThreeRows',
 'MancalaFourRows',
 'MancalaSixRows',
 'MancalaCircular',
 'AlquerqueBoard',
 'AlquerqueBoardWithOneTriangle',
 'AlquerqueBoardWithTwoTriangles',
 'AlquerqueBoardWithFourTriangles',
 'AlquerqueBoardWithEightTriangles',
 'ThreeMensMorrisBoard',
 'ThreeMensMorrisBoardWithTwoTriangles',
 'NineMensMorrisBoard',
 'StarBoard',
 'CrossBoard',
 'KintsBoard',
 'PachisiBoard',
 'FortyStonesWithFourGapsBoard',
 'Track',
 'TrackLoop',
 'TrackOwned',
 'Region',
 'Boardless',
 'Vertex',
 'Cell',
 'Edge',
 'NumPlayableSitesOnBoard',
 'NumColumns',
 'NumRows',
 'NumCorners',
 'NumDirections',
 'NumOrthogonalDirections',
 'NumDiagonalDirections',
 'NumAdjacentDirections',
 'NumOffDiagonalDirections',
 'NumOuterSites',
 'NumInnerSites',
 'NumLayers',
 'NumEdges',
 'NumCells',
 'NumVertices',
 'NumTopSites',
 'NumBottomSites',
 'NumRightSites',
 'NumLeftSites',
 'NumCentreSites',
 'NumConvexCorners',
 'NumConcaveCorners',
 'NumPhasesBoard',
 'Hand',
 'NumContainers',
 'NumPlayableSites',
 'Piece',
 'PieceValue',
 'PieceRotation',
 'Dice',
 'DiceD2',
 'DiceD4',
 'DiceD6',
 'LargePiece',
 'Tile',
 'NumComponentsType',
 'NumComponentsTypePerPlayer',
 'NumDice',
 'Meta',
 'OpeningContract',
 'Repetition',
 'TurnKo',
 'PositionalSuperko',
 'AutoMove',
 'Start',
 'PiecesPlacedOnBoard',
 'PiecesPlacedOutsideBoard',
 'InitialRandomPlacement',
 'InitialScore',
 'InitialCost',
 'NumStartComponentsBoard',
 'NumStartComponentsHand',
 'NumStartComponents',
 'NumStartComponentsBoardPerPlayer',
 'NumStartComponentsHandPerPlayer',
 'NumStartComponentsPerPlayer',
 'Moves',
 'MovesDecision',
 'NoSiteMoves',
 'SwapPlayersDecisionFrequency',
 'PassDecisionFrequency',
 'ProposeDecisionFrequency',
 'SingleSiteMoves',
 'AddDecisionFrequency',
 'PromotionDecisionFrequency',
 'RemoveDecisionFrequency',
 'RotationDecisionFrequency',
 'TwoSitesMoves',
 'StepDecisionFrequency',
 'StepDecisionToEmptyFrequency',
 'StepDecisionToFriendFrequency',
 'StepDecisionToEnemyFrequency',
 'SlideDecisionFrequency',
 'SlideDecisionToEmptyFrequency',
 'SlideDecisionToEnemyFrequency',
 'SlideDecisionToFriendFrequency',
 'LeapDecisionFrequency',
 'LeapDecisionToEmptyFrequency',
 'LeapDecisionToEnemyFrequency',
 'HopDecisionFrequency',
 'HopDecisionMoreThanOneFrequency',
 'HopDecisionEnemyToEmptyFrequency',
 'HopDecisionFriendToEmptyFrequency',
 'HopDecisionFriendToFriendFrequency',
 'HopDecisionEnemyToEnemyFrequency',
 'HopDecisionFriendToEnemyFrequency',
 'FromToDecisionFrequency',
 'FromToDecisionWithinBoardFrequency',
 'FromToDecisionBetweenContainersFrequency',
 'FromToDecisionEmptyFrequency',
 'FromToDecisionEnemyFrequency',
 'FromToDecisionFriendFrequency',
 'SwapPiecesDecisionFrequency',
 'ShootDecisionFrequency',
 'MovesNonDecision',
 'MovesEffects',
 'RollFrequency',
 'ProposeEffectFrequency',
 'AddEffectFrequency',
 'SowFrequency',
 'SowWithEffect',
 'SowCaptureFrequency',
 'SowRemoveFrequency',
 'SowBacktrackingFrequency',
 'SowProperties',
 'SowSkip',
 'SowCW',
 'SowCCW',
 'PromotionEffectFrequency',
 'RemoveEffectFrequency',
 'PushEffectFrequency',
 'FlipFrequency',
 'SetMove',
 'SetNextPlayerFrequency',
 'MoveAgainFrequency',
 'SetValueFrequency',
 'SetCountFrequency',
 'MovesOperators',
 'Priority',
 'ByDieMove',
 'MaxMovesInTurn',
 'MaxDistance',
 'Capture',
 'ReplacementCaptureFrequency',
 'HopCaptureFrequency',
 'HopCaptureMoreThanOneFrequency',
 'DirectionCaptureFrequency',
 'EncloseCaptureFrequency',
 'CustodialCaptureFrequency',
 'InterveneCaptureFrequency',
 'SurroundCaptureFrequency',
 'CaptureSequenceFrequency',
 'Conditions',
 'SpaceConditions',
 'Line',
 'Connection',
 'Group',
 'Contains',
 'Loop',
 'Pattern',
 'Territory',
 'Fill',
 'Distance',
 'MoveConditions',
 'NoMoves',
 'NoMovesMover',
 'NoMovesNext',
 'CanMove',
 'CanNotMove',
 'PieceConditions',
 'NoPiece',
 'NoPieceMover',
 'NoPieceNext',
 'NoTargetPiece',
 'Threat',
 'IsEmpty',
 'IsEnemy',
 'IsFriend',
 'IsPieceAt',
 'LineOfSight',
 'CountPiecesComparison',
 'CountPiecesMoverComparison',
 'CountPiecesNextComparison',
 'ProgressCheck',
 'Directions',
 'AbsoluteDirections',
 'AllDirections',
 'AdjacentDirection',
 'OrthogonalDirection',
 'DiagonalDirection',
 'RotationalDirection',
 'SameLayerDirection',
 'RelativeDirections',
 'ForwardDirection',
 'BackwardDirection',
 'ForwardsDirection',
 'BackwardsDirection',
 'RightwardDirection',
 'RightwardsDirection',
 'ForwardRightDirection',
 'BackwardRightDirection',
 'SameDirection',
 'OppositeDirection',
 'Phase',
 'NumPlayPhase',
 'Scoring',
 'PieceCount',
 'SumDice',
 'SpaceEnd',
 'LineEndFrequency',
 'LineWinFrequency',
 'LineLossFrequency',
 'ConnectionEndFrequency',
 'ConnectionWinFrequency',
 'ConnectionLossFrequency',
 'GroupEndFrequency',
 'GroupWinFrequency',
 'LoopWinFrequency',
 'PatternWinFrequency',
 'TerritoryWinFrequency',
 'CaptureEnd',
 'CheckmateFrequency',
 'CheckmateWinFrequency',
 'NoTargetPieceEndFrequency',
 'NoTargetPieceWinFrequency',
 'EliminatePiecesEndFrequency',
 'EliminatePiecesWinFrequency',
 'EliminatePiecesLossFrequency',
 'EliminatePiecesDrawFrequency',
 'RaceEnd',
 'NoOwnPiecesEndFrequency',
 'NoOwnPiecesWinFrequency',
 'NoOwnPiecesLossFrequency',
 'FillEndFrequency',
 'FillWinFrequency',
 'ReachEndFrequency',
 'ReachWinFrequency',
 'ReachLossFrequency',
 'ReachDrawFrequency',
 'ScoringEndFrequency',
 'ScoringWinFrequency',
 'ScoringLossFrequency',
 'NoMovesEndFrequency',
 'NoMovesWinFrequency',
 'NoMovesLossFrequency',
 'NoMovesDrawFrequency',
 'NoProgressEndFrequency',
 'NoProgressDrawFrequency',
 'DrawFrequency',
 'Misere',
 'DurationActions',
 'DurationMoves',
 'DurationTurns',
 'DurationTurnsStdDev',
 'DurationTurnsNotTimeouts',
 'DecisionMoves',
 'GameTreeComplexity',
 'StateTreeComplexity',
 'BoardCoverageDefault',
 'BoardCoverageFull',
 'BoardCoverageUsed',
 'AdvantageP1',
 'Balance',
 'Completion',
 'Drawishness',
 'Timeouts',
 'OutcomeUniformity',
 'BoardSitesOccupiedAverage',
 'BoardSitesOccupiedMedian',
 'BoardSitesOccupiedMaximum',
 'BoardSitesOccupiedVariance',
 'BoardSitesOccupiedChangeAverage',
 'BoardSitesOccupiedChangeSign',
 'BoardSitesOccupiedChangeLineBestFit',
 'BoardSitesOccupiedChangeNumTimes',
 'BoardSitesOccupiedMaxIncrease',
 'BoardSitesOccupiedMaxDecrease',
 'BranchingFactorAverage',
 'BranchingFactorMedian',
 'BranchingFactorMaximum',
 'BranchingFactorVariance',
 'BranchingFactorChangeAverage',
 'BranchingFactorChangeSign',
 'BranchingFactorChangeLineBestFit',
 'BranchingFactorChangeNumTimesn',
 'BranchingFactorChangeMaxIncrease',
 'BranchingFactorChangeMaxDecrease',
 'DecisionFactorAverage',
 'DecisionFactorMedian',
 'DecisionFactorMaximum',
 'DecisionFactorVariance',
 'DecisionFactorChangeAverage',
 'DecisionFactorChangeSign',
 'DecisionFactorChangeLineBestFit',
 'DecisionFactorChangeNumTimes',
 'DecisionFactorMaxIncrease',
 'DecisionFactorMaxDecrease',
 'MoveDistanceAverage',
 'MoveDistanceMedian',
 'MoveDistanceMaximum',
 'MoveDistanceVariance',
 'MoveDistanceChangeAverage',
 'MoveDistanceChangeSign',
 'MoveDistanceChangeLineBestFit',
 'MoveDistanceChangeNumTimes',
 'MoveDistanceMaxIncrease',
 'MoveDistanceMaxDecrease',
 'PieceNumberAverage',
 'PieceNumberMedian',
 'PieceNumberMaximum',
 'PieceNumberVariance',
 'PieceNumberChangeAverage',
 'PieceNumberChangeSign',
 'PieceNumberChangeLineBestFit',
 'PieceNumberChangeNumTimes',
 'PieceNumberMaxIncrease',
 'PieceNumberMaxDecrease',
 'ScoreDifferenceAverage',
 'ScoreDifferenceMedian',
 'ScoreDifferenceMaximum',
 'ScoreDifferenceVariance',
 'ScoreDifferenceChangeAverage',
 'ScoreDifferenceChangeSign',
 'ScoreDifferenceChangeLineBestFit',
 'ScoreDifferenceMaxIncrease',
 'ScoreDifferenceMaxDecrease',
 'Math',
 'Arithmetic',
 'Operations',
 'Addition',
 'Subtraction',
 'Multiplication',
 'Division',
 'Modulo',
 'Absolute',
 'Exponentiation',
 'Minimum',
 'Maximum',
 'Comparison',
 'Equal',
 'NotEqual',
 'LesserThan',
 'LesserThanOrEqual',
 'GreaterThan',
 'GreaterThanOrEqual',
 'Parity',
 'Even',
 'Odd',
 'Logic',
 'Conjunction',
 'Disjunction',
 'Negation',
 'Set',
 'Union',
 'Intersection',
 'Complement',
 'Algorithmics',
 'ConditionalStatement',
 'ControlFlowStatement',
 'Visual',
 'Style',
 'BoardStyle',
 'GraphStyle',
 'ChessStyle',
 'GoStyle',
 'PenAndPaperStyle',
 'BackgammonStyle',
 'JanggiStyle',
 'XiangqiStyle',
 'ShogiStyle',
 'TableStyle',
 'SurakartaStyle',
 'TaflStyle',
 'NoBoard',
 'ComponentStyle',
 'AnimalComponent',
 'ChessComponent',
 'KingComponent',
 'QueenComponent',
 'KnightComponent',
 'RookComponent',
 'BishopComponent',
 'PawnComponent',
 'FairyChessComponent',
 'PloyComponent',
 'ShogiComponent',
 'XiangqiComponent',
 'StrategoComponent',
 'JanggiComponent',
 'CheckersComponent',
 'BallComponent',
 'TaflComponent',
 'DiscComponent',
 'MarkerComponent',
 'StackType',
 'Stack',
 'Symbols',
 'ShowPieceValue',
 'ShowPieceState',
 'Implementation',
 'State',
 'StateType',
 'StackState',
 'PieceState',
 'SiteState',
 'SetSiteState',
 'VisitedSites',
 'Variable',
 'SetVar',
 'RememberValues',
 'ForgetValues',
 'SetPending',
 'InternalCounter',
 'SetInternalCounter',
 'PlayerValue',
 'Efficiency',
 'CopyContext',
 'Then',
 'ForEachPiece',
 'DoLudeme',
 'Trigger',
 'PlayoutsPerSecond',
 'MovesPerSecond',]

In [26]:
def create_2stgpred(data,modellist_1st,modellist_2nd,fold):
    model1 = modellist_1st[fold]
    model2 = modellist_2nd[fold]
    pred_df = pd.DataFrame(data)
    pred_df["original_index"] = data.index

    y_test_pred = model1.predict(data)

    pred_df["pred"] = y_test_pred
    pred_df["pred_prob"] = np.max(model1.predict_proba(data), axis=1)
    pred_df_1st = pred_df[(pred_df['pred'].isin([0, 1])) & (pred_df['pred_prob'] >= 0.8)]
    pred_df_2st = pred_df[~((pred_df['pred'].isin([0, 1])) & (pred_df['pred_prob'] >= 0.8))]
    X_test_2nd = pred_df_2st.drop(["pred","pred_prob","original_index"], axis=1)

    y_test_pred = model2.predict(X_test_2nd)
    pred_df_2st["pred"] = y_test_pred
    pred_df_1st["pred"] = pred_df_1st["pred"].apply(lambda x: -1 if x == 0 else 1)

    pred_df = pd.concat([pred_df_1st, pred_df_2st])

    df_combine = pred_df.sort_values("original_index")
    return df_combine["pred"].clip(-1,1)

In [27]:
def create_3tarpred(data,modellist_win,modellist_loss,modellist_draw,fold):
    model_win = modellist_win[fold]
    model_loss = modellist_loss[fold]
    model_draw = modellist_draw[fold]
    pred_win = model_win.predict(data)
    pred_loss = model_loss.predict(data)
    pred_draw = model_draw.predict(data)
    pred = (pred_win-pred_loss)/(pred_win+pred_loss+pred_draw)
    return pred.clip(-1,1)

In [28]:
for fold in range(5):
    os.makedirs(f"/tmp/workdir/kaggle/working/nn_models/fold{fold}", exist_ok=True)
    os.makedirs(f"/kaggle/working/nn_models/fold{fold}", exist_ok=True)
    os.system(f'cp -r /kaggle/input/mcts-deeptable1/fold{fold}/* /tmp/workdir/kaggle/working/nn_models/fold{fold}/')
    os.system(f'cp -r /kaggle/input/mcts-deeptable1/fold{fold}/* /kaggle/working/nn_models/fold{fold}/')

In [29]:
def load_model(paths):
    models = []
    for fold in sorted(os.listdir(paths)):
        path = os.path.join(paths, fold)
        for file in os.listdir(path):
            if file.endswith('.h5'):
                models.append(DeepTable.load(path, file))
            elif file.endswith('.cbm'):
                print('Load model from:', path+'/'+file)
                models.append(CatBoostRegressor().load_model(path+'/'+file))
    return models

nn_models = load_model("/kaggle/working/nn_models")

11-18 00:04:20 I deeptables.m.deeptable.py 818 - Load model from: /kaggle/working/nn_models/fold0/dnn_netsafm_netscin_nets.h5.
11-18 00:04:25 I deeptables.m.deeptable.py 818 - Load model from: /kaggle/working/nn_models/fold1/dnn_netsafm_netscin_nets.h5.
11-18 00:04:29 I deeptables.m.deeptable.py 818 - Load model from: /kaggle/working/nn_models/fold2/dnn_netsafm_netscin_nets.h5.
11-18 00:04:32 I deeptables.m.deeptable.py 818 - Load model from: /kaggle/working/nn_models/fold3/dnn_netsafm_netscin_nets.h5.
11-18 00:04:37 I deeptables.m.deeptable.py 818 - Load model from: /kaggle/working/nn_models/fold4/dnn_netsafm_netscin_nets.h5.


In [30]:
scalers = [joblib.load(f'/kaggle/input/mcts-scaler/scaler_fold{i}.pickle') for i in range(5)]

In [31]:
models_simple_cat_0 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_normal_seed0_fold{i}.pickle') for i in range(5)]
models_simple_cat_1 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_normal_seed1_fold{i}.pickle') for i in range(5)]
models_simple_cat_2 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_normal_seed2_fold{i}.pickle') for i in range(5)]

models_simple_lgb_0 = [joblib.load(f'/kaggle/input/mcts-basemodels/lgb_model_seed0_fold{i}.pickle') for i in range(5)]
models_simple_lgb_1 = [joblib.load(f'/kaggle/input/mcts-basemodels/lgb_model_seed1_fold{i}.pickle') for i in range(5)]
models_simple_lgb_2 = [joblib.load(f'/kaggle/input/mcts-basemodels/lgb_model_seed2_fold{i}.pickle') for i in range(5)]

models_unbalance_cat_1st_0 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_1st_seed0_fold{i}.pickle') for i in range(5)]
models_unbalance_cat_2nd_0 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_2nd_seed0_fold{i}.pickle') for i in range(5)]
models_unbalance_cat_1st_1 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_1st_seed1_fold{i}.pickle') for i in range(5)]
models_unbalance_cat_2nd_1 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_2nd_seed1_fold{i}.pickle') for i in range(5)]
models_unbalance_cat_1st_2 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_1st_seed2_fold{i}.pickle') for i in range(5)]
models_unbalance_cat_2nd_2 = [joblib.load(f'/kaggle/input/mcts-basemodels/cat_model_unbalance_2nd_seed2_fold{i}.pickle') for i in range(5)]

models_target_win_0 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_win_seed0_fold{i}.pickle") for i in range(5)]
models_target_loss_0 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_loss_seed0_fold{i}.pickle") for i in range(5)]
models_target_draw_0 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_draw_seed0_fold{i}.pickle") for i in range(5)]
models_target_win_1 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_win_seed1_fold{i}.pickle") for i in range(5)]
models_target_loss_1 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_loss_seed1_fold{i}.pickle") for i in range(5)]
models_target_draw_1 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_draw_seed1_fold{i}.pickle") for i in range(5)]
models_target_win_2 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_win_seed2_fold{i}.pickle") for i in range(5)]
models_target_loss_2 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_loss_seed2_fold{i}.pickle") for i in range(5)]
models_target_draw_2 = [joblib.load(f"/kaggle/input/mcts-basemodels/cat_model_target_draw_seed2_fold{i}.pickle") for i in range(5)]

In [32]:
models=[]

In [33]:
def infer_lgb(data, models):
    pred_list = []
    for fold in tqdm(range(5)):
        if fold == 0:
            continue
        elif fold == 2:
            continue
        # lgb simple
        pred = models_simple_lgb_0[fold].predict(data)
        pred2 = models_simple_lgb_1[fold].predict(data)
        pred3 = models_simple_lgb_2[fold].predict(data)
    
        pred_lgb = (pred*0.4+pred2*0.3+pred3*0.3)*1.1
    
        # catboost simple
        pred = models_simple_cat_0[fold].predict(data)
        pred2 = models_simple_cat_1[fold].predict(data)
        pred3 = models_simple_cat_2[fold].predict(data)
    
        pred_cat = (pred*0.35+pred2*0.35+pred3*0.3)*1.1
    
        # catboost unbalance
    
        pred_unbalance_0 = create_2stgpred(data,models_unbalance_cat_1st_0,models_unbalance_cat_2nd_0,fold)
        pred_unbalance_1 = create_2stgpred(data,models_unbalance_cat_1st_1,models_unbalance_cat_2nd_1,fold)
        pred_unbalance_2 = create_2stgpred(data,models_unbalance_cat_1st_2,models_unbalance_cat_2nd_2,fold)
    
        pred_unbalance = pred_unbalance_0*0.4 + pred_unbalance_1*0.3 + pred_unbalance_2*0.3
        pred_unbalance = pred_unbalance*1.1
    
        # catboost 3target
        pred_target_0 = create_3tarpred(data,models_target_win_0,models_target_loss_0,models_target_draw_0,fold)
        pred_target_1 = create_3tarpred(data,models_target_win_1,models_target_loss_1,models_target_draw_1,fold)
        pred_target_2 = create_3tarpred(data,models_target_win_2,models_target_loss_2,models_target_draw_2,fold)
        pred_target = pred_target_0*0.2 + pred_target_1*0.4 + pred_target_2*0.4
        pred_target = pred_target*1.1
    
        pred_stack = (pred_unbalance*0.2 + pred_cat*0.2 + pred_target*0.5 + pred_lgb*0.1)

        data_copy = data.copy()
        numeric_features = data.select_dtypes(include=['number']).columns
        data_copy[numeric_features] = scalers[fold].transform(data_copy[numeric_features])
        pred_nn = nn_models[fold].predict(data_copy, verbose=1, batch_size=512)
        pred_nn = pred_nn.flatten()*1.1

        pred_sub = (pred_stack * 0.9 + pred_nn*0.1).clip(-1,1)

        pred_list.append(pred_sub)
    return np.mean(pred_list, axis=0).clip(-1,1)


def predict(test, submission):
    
#     test = extract_from_engrule(test,tfidf,svd)
    test = extract_game_name(test)
    test = remove_useless(test,nomean_cols,freq_cols ,duplicate_cols)
    test = get_row_feature(test)
#     test = create_category(test,category_list, addcat_list)
#     test = create_category_sum(test)
    test = split_rule(test)
    test = make_rule_features(test)
    test = process_agent_cols(test)
    test = add_agentcol(test)
    # test = drop_col_almostnomean(test)
    test = feature_domain(test)
    test = normalize_columns(test)
    test = feature_engineering(test)
#     test_data = create_bin(test)
    
    return submission.with_columns(pl.Series('utility_agent1', infer_lgb(test, models)))

inference_server = kaggle_evaluation.mcts_inference_server.MCTSInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
            '/kaggle/input/um-game-playing-strength-of-mcts-variants/test.csv',
            '/kaggle/input/um-game-playing-strength-of-mcts-variants/sample_submission.csv'
        )
    )

I0000 00:00:1731888308.242066     111 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1731888308.368132     111 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1731888308.485140     111 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1731888308.608784     111 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
  0%|          | 0/5 [00:00<?, ?it/s]

11-18 00:05:10 I deeptables.m.deeptable.py 685 - Perform prediction...
11-18 00:05:10 I deeptables.m.preprocessor.py 244 - Transform [X]...
11-18 00:05:10 I deeptables.m.preprocessor.py 251 - transform_X taken 0.1230013370513916s
11-18 00:05:10 I deeptables.m.deepmodel.py 130 - Performing predictions...
11-18 00:05:10 I deeptables.u.dataset_generator.py 250 - create dataset generator with _TFDGForPandas, batch_size=512, shuffle=False, drop_remainder=False
1/1 [==============================] - 2s 2s/step
11-18 00:05:12 I deeptables.m.deeptable.py 559 - predict_proba taken 2.3683664798736572s


 40%|████      | 2/5 [00:03<00:04,  1.51s/it]

11-18 00:05:13 I deeptables.m.deeptable.py 685 - Perform prediction...
11-18 00:05:13 I deeptables.m.preprocessor.py 244 - Transform [X]...
11-18 00:05:13 I deeptables.m.preprocessor.py 251 - transform_X taken 0.12211465835571289s
11-18 00:05:13 I deeptables.m.deepmodel.py 130 - Performing predictions...
11-18 00:05:13 I deeptables.u.dataset_generator.py 250 - create dataset generator with _TFDGForPandas, batch_size=512, shuffle=False, drop_remainder=False
1/1 [==============================] - 2s 2s/step
11-18 00:05:15 I deeptables.m.deeptable.py 559 - predict_proba taken 2.292492151260376s


 80%|████████  | 4/5 [00:05<00:01,  1.47s/it]

11-18 00:05:16 I deeptables.m.deeptable.py 685 - Perform prediction...
11-18 00:05:16 I deeptables.m.preprocessor.py 244 - Transform [X]...
11-18 00:05:16 I deeptables.m.preprocessor.py 251 - transform_X taken 0.12147212028503418s
11-18 00:05:16 I deeptables.m.deepmodel.py 130 - Performing predictions...
11-18 00:05:16 I deeptables.u.dataset_generator.py 250 - create dataset generator with _TFDGForPandas, batch_size=512, shuffle=False, drop_remainder=False
1/1 [==============================] - 2s 2s/step
11-18 00:05:18 I deeptables.m.deeptable.py 559 - predict_proba taken 2.2622830867767334s


100%|██████████| 5/5 [00:08<00:00,  1.76s/it]
